In [4]:
import requests
import zipfile
import os
import pandas as pd
from datetime import datetime, timedelta
import pytz


def clean_column_names(df, prefixes):
    new_columns = []
    for col in df.columns:
        cleaned_col = col
        for prefix in prefixes:
            if cleaned_col.startswith(prefix):
                cleaned_col = cleaned_col[len(prefix):]
                break
        new_columns.append(cleaned_col)
    df.columns = new_columns
    return df


sepas = {
    "lunes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/0a9069a9-06e8-4f98-874d-da5578693290/download/sepa_lunes.zip",
    "martes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/9dc06241-cc83-44f4-8e25-c9b1636b8bc8/download/sepa_martes.zip",
    "miercoles": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/1e92cd42-4f94-4071-a165-62c4cb2ce23c/download/sepa_miercoles.zip",
    "jueves": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/d076720f-a7f0-4af8-b1d6-1b99d5a90c14/download/sepa_jueves.zip",
    "viernes": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/91bc072a-4726-44a1-85ec-4a8467aad27e/download/sepa_viernes.zip",
    "sabado": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/b3c3da5d-213d-41e7-8d74-f23fda0a3c30/download/sepa_sabado.zip",
    "domingo": "https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/f8e75128-515a-436e-bf8d-5c63a62f2005/download/sepa_domingo.zip"
}


# Define Buenos Aires timezone
buenos_aires_tz = pytz.timezone('America/Argentina/Buenos_Aires')

# Get current time in Buenos Aires
now_ba = datetime.now(buenos_aires_tz)

# Determine the date for the data to fetch
if now_ba.hour >= 15:  # 3 PM (15:00) cutoff
    target_date = now_ba.date()
elif now_ba.hour < 15:
    target_date = now_ba.date() - timedelta(days=1)

# Get the day of the week (0=Monday, 6=Sunday)
day_of_week_num = target_date.weekday()

day_names = [
    "lunes",
    "martes",
    "miercoles",
    "jueves",
    "viernes",
    "sabado",
    "domingo",
]

# Get the corresponding day name
target_day_name = day_names[day_of_week_num]

# Initialize a flag to control processing
should_process_zip = False

# Ensure zip_file_name is always set
zip_file_name = f"sepa_{target_day_name}.zip"

# First attempt to get the URL
current_zip_url = sepas.get(target_day_name)

if not current_zip_url:
    print(f"Error: No ZIP URL found for {target_day_name} in the URL dictionary.")
    if os.path.exists(zip_file_name):
        print(f"However, ZIP file '{zip_file_name}' found locally. Attempting to use local file.")
        should_process_zip = True # Trust existing file for now, will validate before extraction
    else:
        print(f"And '{zip_file_name}' not found locally. Skipping file extraction and processing.")
else: # current_zip_url is valid
    if os.path.exists(zip_file_name):
        print(f"ZIP file '{zip_file_name}' already exists locally. Checking validity...")
        try:
            # Attempt to open as a zip file to validate it
            with zipfile.ZipFile(zip_file_name, 'r') as zip_test:
                zip_test.testzip() # This method checks the integrity of the archive
            print(f"Local ZIP file '{zip_file_name}' is valid. Skipping download.")
            should_process_zip = True
        except zipfile.BadZipFile:
            print(f"Warning: Local file '{zip_file_name}' is corrupt or not a valid ZIP file. Deleting and attempting re-download.")
            os.remove(zip_file_name) # Delete corrupt file
            # Fall through to download logic below
        except Exception as e:
            print(f"Error validating local ZIP file '{zip_file_name}': {e}. Deleting and attempting re-download.")
            os.remove(zip_file_name)
            # Fall through to download logic below

    # If no valid local file was found (either didn't exist, or was corrupt and deleted), then download
    if not should_process_zip:
        # Add headers to mimic a browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }

        try:
            print(f"Attempting to download '{zip_file_name}' from {current_zip_url}.")
            response = requests.get(current_zip_url, headers=headers)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

            with open(zip_file_name, 'wb') as f:
                f.write(response.content)

            print(f"ZIP file '{zip_file_name}' downloaded successfully.")
            should_process_zip = True # Set to true if download was successful
        except requests.exceptions.RequestException as e:
            print(f"Error downloading ZIP file: {e}")
            print(f"Please try to manually download '{zip_file_name}' from the website and upload it to your Colab environment.")
            should_process_zip = False # Download failed, do not process

# Now, the main processing block, proceed only if should_process_zip is True and the file exists
if should_process_zip and os.path.exists(zip_file_name):
    # Final verification of the ZIP file before extraction to catch issues with newly downloaded files
    try:
        with zipfile.ZipFile(zip_file_name, 'r') as zip_test_extract:
            zip_test_extract.testzip() # This also raises BadZipFile if corrupt
    except zipfile.BadZipFile:
        print(f"Critical Error: '{zip_file_name}' is not a valid ZIP file even after (re)download/local check. Skipping extraction and further processing.")
        if os.path.exists(zip_file_name):
            os.remove(zip_file_name)
            print(f"Removed corrupt file '{zip_file_name}'.")
        should_process_zip = False
    except Exception as e:
        print(f"Critical Error: An unexpected error occurred while verifying '{zip_file_name}': {e}. Skipping extraction and further processing.")
        if os.path.exists(zip_file_name):
            os.remove(zip_file_name)
            print(f"Removed problematic file '{zip_file_name}'.")
        should_process_zip = False

if should_process_zip:
    # Directory to extract the contents to
    extract_dir = target_date.strftime('%Y-%m-%d')
    os.makedirs(extract_dir, exist_ok=True)

    # Create a ZipFile object and extract all contents
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Contents of '{zip_file_name}' extracted to '{os.path.abspath(extract_dir)}' successfully.")

    # Determine the actual directory containing the files to process
    # This handles cases where the main ZIP extracts into a single top-level folder
    actual_content_dir = extract_dir
    extracted_items = os.listdir(extract_dir)
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(extract_dir, extracted_items[0])):
        actual_content_dir = os.path.join(extract_dir, extracted_items[0])

    # Now, nested_zip_search_dir should point to the directory that actually contains the zips to be extracted
    nested_zip_search_dir = actual_content_dir

    # List all files in the nested_zip_search_dir
    all_files_in_nested_dir = os.listdir(nested_zip_search_dir)

    # Filter for .zip files (these are the inner zips)
    nested_zip_files = [f for f in all_files_in_nested_dir if f.endswith('.zip')]

    print(f"Found {len(nested_zip_files)} nested ZIP files in '{nested_zip_search_dir}':")
    for nested_zip_file_name in nested_zip_files:
        nested_zip_file_path = os.path.join(nested_zip_search_dir, nested_zip_file_name)

        # Create a sub-directory for each nested zip's extraction to avoid clutter
        # Extract into a directory within actual_content_dir
        extraction_sub_dir = os.path.join(actual_content_dir, nested_zip_file_name.replace('.zip', ''))
        os.makedirs(extraction_sub_dir, exist_ok=True)

        try:
            with zipfile.ZipFile(nested_zip_file_path, 'r') as nested_zip_ref:
                nested_zip_ref.extractall(extraction_sub_dir)
            print(f"  - Extracted '{nested_zip_file_name}' to '{extraction_sub_dir}'")
        except zipfile.BadZipFile:
            print(f"  - Warning: '{nested_zip_file_name}' is not a valid ZIP file and could not be extracted.")
            if os.path.exists(nested_zip_file_path):
                os.remove(nested_zip_file_path)
                print(f"    Removed corrupt nested file '{nested_zip_file_name}'.")
        except Exception as e:
            print(f"  - Error extracting '{nested_zip_file_name}': {e}")

    print("Finished extracting all identified nested ZIP files.")

    csv_files_found = []
    search_dir = actual_content_dir # Scan for CSVs starting from the actual content directory

    # Walk through the directory and its subdirectories
    for root, dirs, files in os.walk(search_dir):
        for file in files:
            if file.endswith('.csv'):
                csv_files_found.append(os.path.join(root, file))

    print(f"Found {len(csv_files_found)} CSV files:")


    # Initialize lists to store categorized files and their sources
    productos_files = []
    sucursales_files = []
    comercio_files = []

    # Iterate through each file path in the csv_files_found list
    for file_path in csv_files_found:
        # Extract the base name of the file (e.g., 'productos.csv')
        file_name = os.path.basename(file_path)
        # Extract the directory path of the file (the parent directory)
        file_source = os.path.dirname(file_path)

        # Categorize the file based on its name and append to the corresponding list
        if 'productos.csv' in file_name:
            productos_files.append((file_path, file_source))
        elif 'sucursales.csv' in file_name:
            sucursales_files.append((file_path, file_source))
        elif 'comercio.csv' in file_name:
            comercio_files.append((file_path, file_source))
        else:
            print(f"Warning: Uncategorized CSV file found: {file_path}")

    print(f"Total productos files found: {len(productos_files)}")
    print(f"Total sucursales files found: {len(sucursales_files)}")
    print(f"Total comercio files found: {len(comercio_files)}")
else:
    print("Skipping file extraction and processing due to unresolvable issues (download failed or no valid local file found).")

ZIP file 'sepa_jueves.zip' already exists locally. Checking validity...
Local ZIP file 'sepa_jueves.zip' is valid. Skipping download.
Contents of 'sepa_jueves.zip' extracted to 'c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29' successfully.
Found 17 nested ZIP files in '2026-10-29\2026-10-29':
  - Extracted 'sepa_1_comercio-sepa-12_2026-10-29_09-05-11.zip' to '2026-10-29\2026-10-29\sepa_1_comercio-sepa-12_2026-10-29_09-05-11'
  - Extracted 'sepa_1_comercio-sepa-15_2026-10-29_09-05-11.zip' to '2026-10-29\2026-10-29\sepa_1_comercio-sepa-15_2026-10-29_09-05-11'
  - Extracted 'sepa_1_comercio-sepa-20_2026-10-29_09-05-11.zip' to '2026-10-29\2026-10-29\sepa_1_comercio-sepa-20_2026-10-29_09-05-11'
  - Extracted 'sepa_1_comercio-sepa-21_2026-10-29_09-05-11.zip' to '2026-10-29\2026-10-29\sepa_1_comercio-sepa-21_2026-10-29_09-05-11'
  - Extracted 'sepa_1_comercio-sepa-23_2026-10-29_09-05-11.zip' to '2026-10-29\2026-10-29\sepa_1_comercio-sepa-23_2026-10-29_09-05-11'
  - Ex

In [5]:
import numpy as np

def robust_load_csv_simplified(file_path, sep, dtype, target_cols=None):
    """
    Loads a CSV file simply, cleans column names, and extracts only the available target columns
    to avoid memory crashes while tolerating files with missing or differently-named columns.
    """
    try:
        # Load the CSV file. We don't use 'usecols' inside read_csv to avoid strict key errors.
        df = pd.read_csv(file_path, sep=sep, dtype=dtype, low_memory=False)
        if df.empty:
            return df

        # Clean column headers (strip leading/trailing whitespaces)
        df.columns = df.columns.str.strip()

        # Clean footer metadata rows
        first_col = df.columns[0]
        mask_footer = df[first_col].astype(str).str.contains('Última|actualización|actualizacion', case=False, na=True)
        df = df[~mask_footer]
        df = df.dropna(how='all')

        # Filter only the columns that actually exist in this file
        if target_cols:
            existing_target_cols = [col for col in target_cols if col in df.columns]
            df = df[existing_target_cols]

        return df
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return pd.DataFrame()

# Target columns matching the user's requirements (both with and without prefix for compatibility)
productos_cols = [
    'id_comercio', 'id_bandera', 'id_sucursal', 'id_producto',
    'productos_descripcion', 'productos_precio_lista',
    'productos_precio_unitario_promo1', 'productos_leyenda_promo1',
    'productos_precio_unitario_promo2', 'productos_leyenda_promo2',
    'descripcion', 'precio_lista', 'precio_unitario_promo1',
    'leyenda_promo1', 'precio_unitario_promo2', 'leyenda_promo2'
]

all_productos_dfs = []
for file_path, file_source in productos_files:
    df = robust_load_csv_simplified(
        file_path,
        sep='|',
        dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str, 'id_producto': float},
        target_cols=productos_cols
    )
    if not df.empty:
        df['file_source'] = file_source
        all_productos_dfs.append(df)

if all_productos_dfs:
    all_productos_df = pd.concat(all_productos_dfs, ignore_index=True)
    productos_prefixes = ['productos_']
    all_productos_df.columns = all_productos_df.columns.str.strip()
    all_productos_df = clean_column_names(all_productos_df, productos_prefixes)
    print(f"Loaded products successfully. Columns present: {list(all_productos_df.columns)}")
    print(f"Current memory usage: {all_productos_df.memory_usage().sum() / 1e6:.2f} MB")
else:
    print("No productos files were loaded.")

# Loading sucursales and comercio
all_sucursales_dfs = []
for file_path, file_source in sucursales_files:
    df = robust_load_csv_simplified(
        file_path,
        sep='|',
        dtype={'id_comercio': str, 'id_bandera': str, 'id_sucursal': str}
    )
    if not df.empty:
        df['file_source'] = file_source
        all_sucursales_dfs.append(df)

if all_sucursales_dfs:
    all_sucursales_df = pd.concat(all_sucursales_dfs, ignore_index=True)
    sucursales_prefixes = ['sucursales_']
    all_sucursales_df.columns = all_sucursales_df.columns.str.strip()
    all_sucursales_df = clean_column_names(all_sucursales_df, sucursales_prefixes)
else:
    print("No sucursales files were loaded.")

all_comercio_dfs = []
for file_path, file_source in comercio_files:
    df = robust_load_csv_simplified(
        file_path,
        sep='|',
        dtype={'id_comercio': str, 'id_bandera': str}
    )
    if not df.empty:
        df['file_source'] = file_source
        all_comercio_dfs.append(df)

if all_comercio_dfs:
    all_comercio_df = pd.concat(all_comercio_dfs, ignore_index=True)
    comercio_prefixes = ['comercio_']
    all_comercio_df.columns = all_comercio_df.columns.str.strip()
    all_comercio_df = clean_column_names(all_comercio_df, comercio_prefixes)
else:
    print("No comercio files were loaded.")

Loaded products successfully. Columns present: ['id_comercio', 'id_bandera', 'id_sucursal', 'id_producto', 'descripcion', 'precio_lista', 'precio_unitario_promo1', 'leyenda_promo1', 'precio_unitario_promo2', 'leyenda_promo2', 'file_source']
Current memory usage: 1256.28 MB


In [7]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("SEPA-Price-History")
    .getOrCreate()
)

print("Spark:", spark.version)


Spark: 4.2.0


In [9]:

from pathlib import Path

project_dir = Path.cwd()

product_files = list(
    project_dir.rglob("productos.csv")
)

print("Product files found:", len(product_files))

for file in product_files[:10]:
    print(file)


Product files found: 48
c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29\2026-10-29\sepa_1_comercio-sepa-12_2026-10-29_09-05-11\productos.csv
c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29\2026-10-29\sepa_1_comercio-sepa-15_2026-10-29_09-05-11\productos.csv
c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29\2026-10-29\sepa_1_comercio-sepa-20_2026-10-29_09-05-11\productos.csv
c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29\2026-10-29\sepa_1_comercio-sepa-21_2026-10-29_09-05-11\productos.csv
c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29\2026-10-29\sepa_1_comercio-sepa-23_2026-10-29_09-05-11\productos.csv
c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29\2026-10-29\sepa_1_comercio-sepa-3_2026-10-29_09-05-11\productos.csv
c:\Users\Fernando Suarez\Projects\fernandosuarez89.github.io\2026-10-29\2026-10-29\sepa_1_comercio-sepa-47_2026-10-29_09-05-11\prod

In [10]:

if not product_files:
    raise FileNotFoundError(
        "No productos.csv found. Run the existing SEPA extraction first."
    )

csv_path = str(product_files[0])

df_products = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .csv(csv_path)
)

df_products.printSchema()
df_products.show(5, truncate=False)


root
 |-- id_comercio: string (nullable = true)
 |-- id_bandera: string (nullable = true)
 |-- id_sucursal: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- productos_ean: string (nullable = true)
 |-- productos_descripcion: string (nullable = true)
 |-- productos_cantidad_presentacion: string (nullable = true)
 |-- productos_unidad_medida_presentacion: string (nullable = true)
 |-- productos_marca: string (nullable = true)
 |-- productos_precio_lista: string (nullable = true)
 |-- productos_precio_referencia: string (nullable = true)
 |-- productos_cantidad_referencia: string (nullable = true)
 |-- productos_unidad_medida_referencia: string (nullable = true)
 |-- productos_precio_unitario_promo1: string (nullable = true)
 |-- productos_leyenda_promo1: string (nullable = true)
 |-- productos_precio_unitario_promo2: string (nullable = true)
 |-- productos_leyenda_promo2: string (nullable = true)

+-----------+----------+-----------+-------------+-------------+---

In [11]:

df_prices = df_products.select(
    "id_comercio",
    "id_bandera",
    "id_sucursal",
    "id_producto",
    F.col("productos_precio_lista")
        .cast("decimal(18,2)").alias("precio_lista"),
    F.col("productos_precio_unitario_promo1")
        .cast("decimal(18,2)").alias("precio_promo1"),
    F.col("productos_precio_unitario_promo2")
        .cast("decimal(18,2)").alias("precio_promo2")
)

df_prices.printSchema()
df_prices.show(5, truncate=False)


root
 |-- id_comercio: string (nullable = true)
 |-- id_bandera: string (nullable = true)
 |-- id_sucursal: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- precio_lista: decimal(18,2) (nullable = true)
 |-- precio_promo1: decimal(18,2) (nullable = true)
 |-- precio_promo2: decimal(18,2) (nullable = true)

+-----------+----------+-----------+-------------+------------+-------------+-------------+
|id_comercio|id_bandera|id_sucursal|id_producto  |precio_lista|precio_promo1|precio_promo2|
+-----------+----------+-----------+-------------+------------+-------------+-------------+
|12         |1         |108        |7791333000449|10145.00    |NULL         |NULL         |
|12         |1         |109        |7791333000449|10145.00    |NULL         |NULL         |
|12         |1         |110        |7791333000449|10145.00    |NULL         |NULL         |
|12         |1         |111        |7791333000449|10145.00    |NULL         |NULL         |
|12         |1         |

In [14]:

from pathlib import Path
from datetime import date
from collections import defaultdict
import re

project_dir = Path.cwd()
product_files = list(project_dir.rglob("productos.csv"))

publications = defaultdict(list)

for path in product_files:
    relative_path = path.relative_to(project_dir)

    # Extract the date from the SEPA merchant folder
    match = re.search(
        r"sepa_\d+_comercio-sepa-\d+_(\d{4}-\d{2}-\d{2})_",
        path.parent.name
    )

    if match:
        publication_date = date.fromisoformat(match.group(1))
        publications[publication_date].append(str(path))

publications = dict(sorted(publications.items()))

for publication_date, files in publications.items():
    print(f"{publication_date}: {len(files)} merchant files")

print("Total publications:", len(publications))
print("Total CSV files:", sum(map(len, publications.values())))


2026-10-26: 16 merchant files
2026-10-27: 16 merchant files
2026-10-29: 16 merchant files
Total publications: 3
Total CSV files: 48


In [15]:

oldest_date = min(publications)

oldest_files = publications[oldest_date]

print("Oldest publication:", oldest_date)
print("Merchant files:", len(oldest_files))

df_products = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .csv(oldest_files)
)

df_products.printSchema()
df_products.show(5, truncate=False)


Oldest publication: 2026-10-26
Merchant files: 16
root
 |-- id_comercio: string (nullable = true)
 |-- id_bandera: string (nullable = true)
 |-- id_sucursal: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- productos_ean: string (nullable = true)
 |-- productos_descripcion: string (nullable = true)
 |-- productos_cantidad_presentacion: string (nullable = true)
 |-- productos_unidad_medida_presentacion: string (nullable = true)
 |-- productos_marca: string (nullable = true)
 |-- productos_precio_lista: string (nullable = true)
 |-- productos_precio_referencia: string (nullable = true)
 |-- productos_cantidad_referencia: string (nullable = true)
 |-- productos_unidad_medida_referencia: string (nullable = true)
 |-- productos_precio_unitario_promo1: string (nullable = true)
 |-- productos_leyenda_promo1: string (nullable = true)
 |-- productos_precio_unitario_promo2: string (nullable = true)
 |-- productos_leyenda_promo2: string (nullable = true)

+-----------+----

In [16]:

business_key = [
    "id_comercio",
    "id_bandera",
    "id_sucursal",
    "id_producto"
]

duplicates = (
    df_products
    .groupBy(*business_key)
    .count()
    .filter("count > 1")
)

print("Duplicate keys:", duplicates.count())
print("Total rows:", df_products.count())
print("Distinct merchants:", df_products.select("id_comercio").distinct().count())


Duplicate keys: 50
Total rows: 14599511
Distinct merchants: 31


In [17]:

from datetime import date
from pyspark.sql import functions as F

oldest_date = min(publications)

df_products = (
    spark.read
    .option("header", "true")
    .option("sep", "|")
    .csv(publications[oldest_date])
)

business_key = [
    "id_comercio",
    "id_bandera",
    "id_sucursal",
    "id_producto"
]

print("Publication:", oldest_date)
print("Total rows:", df_products.count())
print("Distinct merchants:",
      df_products.select("id_comercio").distinct().count())

duplicates = (
    df_products
    .groupBy(*business_key)
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate keys:", duplicates.count())


Publication: 2026-10-26
Total rows: 14599511
Distinct merchants: 31
Duplicate keys: 50


In [18]:

from pyspark.sql import functions as F

price_columns = [
    "productos_precio_lista",
    "productos_precio_unitario_promo1",
    "productos_precio_unitario_promo2"
]

duplicate_details = (
    df_products
    .join(
        duplicates.select(*business_key),
        on=business_key,
        how="inner"
    )
    .select(
        *business_key,
        *price_columns
    )
    .orderBy(*business_key)
)

duplicate_details.show(30, truncate=False)


+-----------+----------+-----------+-------------+----------------------+--------------------------------+--------------------------------+
|id_comercio|id_bandera|id_sucursal|id_producto  |productos_precio_lista|productos_precio_unitario_promo1|productos_precio_unitario_promo2|
+-----------+----------+-----------+-------------+----------------------+--------------------------------+--------------------------------+
|5          |1         |1          |0020800722452|859.00                |NULL                            |NULL                            |
|5          |1         |1          |0020800722452|859.00                |NULL                            |NULL                            |
|5          |1         |1          |0041789001918|2245.00               |NULL                            |NULL                            |
|5          |1         |1          |0041789001918|2245.00               |NULL                            |NULL                            |
|5          |1      

In [19]:

duplicate_summary = (
    duplicate_details
    .groupBy(*business_key)
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            F.struct(*[
                F.col(c) for c in price_columns
            ])
        ).alias("distinct_price_combinations")
    )
)

duplicate_summary.groupBy(
    "distinct_price_combinations"
).count().orderBy(
    "distinct_price_combinations"
).show()


+---------------------------+-----+
|distinct_price_combinations|count|
+---------------------------+-----+
|                          1|   49|
+---------------------------+-----+



In [21]:
duplicate_details.show(60, truncate=False)

+-----------+----------+-----------+-------------+----------------------+--------------------------------+--------------------------------+
|id_comercio|id_bandera|id_sucursal|id_producto  |productos_precio_lista|productos_precio_unitario_promo1|productos_precio_unitario_promo2|
+-----------+----------+-----------+-------------+----------------------+--------------------------------+--------------------------------+
|5          |1         |1          |0020800722452|859.00                |NULL                            |NULL                            |
|5          |1         |1          |0020800722452|859.00                |NULL                            |NULL                            |
|5          |1         |1          |0041789001918|2245.00               |NULL                            |NULL                            |
|5          |1         |1          |0041789001918|2245.00               |NULL                            |NULL                            |
|5          |1      

In [22]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

business_key = [
    "id_comercio",
    "id_bandera",
    "id_sucursal",
    "id_producto"
]

window = (
    Window
    .partitionBy(*business_key)
    .orderBy(
        F.col("productos_ean").cast("int").desc_nulls_last()
    )
)

df_products_clean = (
    df_products
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)


In [23]:

print("Original rows:", df_products.count())
print("Clean rows:", df_products_clean.count())

remaining_duplicates = (
    df_products_clean
    .groupBy(*business_key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Remaining duplicate keys:", remaining_duplicates)


Original rows: 14599511
Clean rows: 14599461
Remaining duplicate keys: 0


In [25]:

from pyspark.sql import functions as F

publication_date = "2026-10-26"

df_history_initial = (
    df_products_clean
    .select(
        "id_comercio",
        "id_bandera",
        "id_sucursal",
        "id_producto",
        F.col("productos_ean").cast("int").alias("productos_ean"),
        F.col("productos_precio_lista")
            .cast("decimal(18,2)").alias("precio_lista"),
        F.col("productos_precio_unitario_promo1")
            .cast("decimal(18,2)").alias("precio_promo1"),
        F.col("productos_precio_unitario_promo2")
            .cast("decimal(18,2)").alias("precio_promo2")
    )
    .withColumn("valid_from", F.to_date(F.lit(publication_date)))
    .withColumn("valid_to", F.lit(None).cast("date"))
    .withColumn("last_seen_date", F.to_date(F.lit(publication_date)))
    .withColumn("is_current", F.lit(True))
)

df_history_initial.printSchema()
df_history_initial.show(5, truncate=False)


root
 |-- id_comercio: string (nullable = true)
 |-- id_bandera: string (nullable = true)
 |-- id_sucursal: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- productos_ean: integer (nullable = true)
 |-- precio_lista: decimal(18,2) (nullable = true)
 |-- precio_promo1: decimal(18,2) (nullable = true)
 |-- precio_promo2: decimal(18,2) (nullable = true)
 |-- valid_from: date (nullable = true)
 |-- valid_to: date (nullable = true)
 |-- last_seen_date: date (nullable = true)
 |-- is_current: boolean (nullable = false)

+-----------+----------+-----------+-------------+-------------+------------+-------------+-------------+----------+--------+--------------+----------+
|id_comercio|id_bandera|id_sucursal|id_producto  |productos_ean|precio_lista|precio_promo1|precio_promo2|valid_from|valid_to|last_seen_date|is_current|
+-----------+----------+-----------+-------------+-------------+------------+-------------+-------------+----------+--------+--------------+----------+


In [26]:

from pathlib import Path
import time

history_path = str(
    Path.cwd() / "data" / "price_history" / "v001"
)

start = time.perf_counter()

(
    df_history_initial
    .write
    .mode("errorifexists")
    .option("compression", "snappy")
    .parquet(history_path)
)

print(f"Write completed in {time.perf_counter() - start:.1f} seconds")
print("Saved to:", history_path)


Py4JJavaError: An error occurred while calling o317.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: C:\hadoop\bin\winutils.exe -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:790)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:299)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:315)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalF